In [31]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gc
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.compose import ColumnTransformer
from my_functions import *

from sklearn.ensemble import RandomForestClassifier

sns.set_theme()
gc.enable()

# Main Data

In [2]:
main_data=pd.read_csv('initial_clean.csv', keep_default_na=False, na_values='')
main_data.drop(['DAYS_BIRTH', 'DAYS_EMPLOYED', 
                'DAYS_ID_PUBLISH', 'DAYS_LAST_PHONE_CHANGE', 
                'DAYS_REGISTRATION'], axis=1, inplace=True)
main_data.shape

(307511, 43)

### Dealing with Outliers

In [3]:
'''
contains all outlier marker columns
'''
outlier_frame=main_data.filter(regex='outlier', axis=1).copy()
outlier_frame['SK_ID_CURR']=main_data['SK_ID_CURR']
outlier_frame['TARGET']=main_data['TARGET']
outlier_frame.shape

(307511, 8)

In [4]:
# plot_all_encoded_vs_target(outlier_frame)

In [5]:
outlier_frame.columns

Index(['AMT_CREDIT_outlier', 'AMT_INCOME_TOTAL_outlier', 'AMT_ANNUITY_outlier',
       'AMT_GOODS_PRICE_outlier', 'DAYS_REGISTRATION_outlier',
       'DAYS_EMPLOYED_outlier', 'SK_ID_CURR', 'TARGET'],
      dtype='object')

In [6]:
clean1=main_data.loc[(main_data['AMT_CREDIT_outlier']==0)&
              (main_data['AMT_INCOME_TOTAL_outlier']==0)&
              (main_data['AMT_ANNUITY_outlier']==0)&
              (main_data['AMT_GOODS_PRICE_outlier']==0)&
              (main_data['DAYS_REGISTRATION_outlier']==0)&
              (main_data['DAYS_EMPLOYED_outlier']==0), :].copy()
clean1.drop(labels=outlier_frame.columns.difference(['SK_ID_CURR', 'TARGET']), axis=1, inplace=True)
clean1.loc[clean1['NAME_FAMILY_STATUS'].isnull(), 'NAME_FAMILY_STATUS']=clean1['NAME_FAMILY_STATUS'].mode().values[0]
clean1.shape

(306295, 37)

### Scaling

In [7]:
clean1.columns.sort_values()

Index(['AMT_ANNUITY', 'AMT_CREDIT', 'AMT_GOODS_PRICE', 'AMT_INCOME_TOTAL',
       'CODE_GENDER', 'EMERGENCYSTATE_MODE', 'EXT_SOURCE_1', 'EXT_SOURCE_2',
       'EXT_SOURCE_3', 'FLAG_DOCUMENT_3', 'FLAG_DOCUMENT_6', 'FLAG_EMP_PHONE',
       'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'FLAG_PHONE', 'FLAG_WORK_PHONE',
       'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'LIVE_CITY_NOT_WORK_CITY',
       'NAME_CONTRACT_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
       'NAME_HOUSING_TYPE', 'NAME_INCOME_TYPE', 'NAME_TYPE_SUITE',
       'OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 'REG_CITY_NOT_LIVE_CITY',
       'REG_CITY_NOT_WORK_CITY', 'SK_ID_CURR', 'TARGET', 'WALLSMATERIAL_MODE',
       'YEARS_BIRTH', 'YEARS_EMPLOYED', 'YEARS_ID_PUBLISH',
       'YEARS_LAST_PHONE_CHANGE', 'YEARS_REGISTRATION'],
      dtype='object')

In [8]:
ct=ColumnTransformer(transformers=[('numerical', StandardScaler(), ['AMT_ANNUITY', 'AMT_CREDIT', 'AMT_INCOME_TOTAL', 'AMT_GOODS_PRICE',
                                                                    'YEARS_BIRTH', 'YEARS_ID_PUBLISH', 'YEARS_REGISTRATION', 
                                                                    'YEARS_EMPLOYED', 'YEARS_LAST_PHONE_CHANGE'])],
                     remainder='passthrough')
clean2=pd.DataFrame(ct.fit_transform(clean1))
clean2.columns=ct.get_feature_names_out()
clean2.rename(columns=lambda x: x[11:], inplace=True)
clean2.shape

(306295, 37)

### Imputing Null Values and Encoding Categorical Variables

In [10]:
clean2.columns

Index(['AMT_ANNUITY', 'AMT_CREDIT', 'AMT_INCOME_TOTAL', 'AMT_GOODS_PRICE',
       'YEARS_BIRTH', 'YEARS_ID_PUBLISH', 'YEARS_REGISTRATION',
       'YEARS_EMPLOYED', 'YEARS_LAST_PHONE_CHANGE', 'EXT_SOURCE_1',
       'EXT_SOURCE_2', 'EXT_SOURCE_3', 'TARGET', 'SK_ID_CURR', 'CODE_GENDER',
       'EMERGENCYSTATE_MODE', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
       'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'NAME_CONTRACT_TYPE',
       'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE',
       'NAME_INCOME_TYPE', 'NAME_TYPE_SUITE', 'OCCUPATION_TYPE',
       'ORGANIZATION_TYPE', 'WALLSMATERIAL_MODE', 'FLAG_DOCUMENT_3',
       'FLAG_DOCUMENT_6', 'FLAG_EMP_PHONE', 'FLAG_PHONE', 'FLAG_WORK_PHONE',
       'LIVE_CITY_NOT_WORK_CITY', 'REG_CITY_NOT_LIVE_CITY',
       'REG_CITY_NOT_WORK_CITY'],
      dtype='object')

In [26]:
ct2=ColumnTransformer(transformers=[('numerical', SimpleImputer(strategy='median'), ['AMT_ANNUITY', 'AMT_CREDIT', 'AMT_INCOME_TOTAL', 
                                                                                   'AMT_GOODS_PRICE', 'YEARS_BIRTH', 'YEARS_ID_PUBLISH', 
                                                                                   'YEARS_REGISTRATION', 'YEARS_EMPLOYED', 'YEARS_LAST_PHONE_CHANGE', 
                                                                                   'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']),
                                    
                                    ('encodings', OneHotEncoder(sparse_output=False), ['CODE_GENDER', 'EMERGENCYSTATE_MODE', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
                                                                    'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'NAME_CONTRACT_TYPE',
                                                                    'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE',
                                                                    'NAME_INCOME_TYPE', 'NAME_TYPE_SUITE', 'OCCUPATION_TYPE',
                                                                    'ORGANIZATION_TYPE', 'WALLSMATERIAL_MODE'])],
                      remainder='passthrough', n_jobs=-1)

clean3=pd.DataFrame(ct2.fit_transform(clean2))
clean3.columns=ct2.get_feature_names_out()
clean3.rename(columns=lambda x: x[11:], inplace=True)
clean3.shape

(306295, 159)